# What we are building
## **An AI Research Paper Assistant**

# **MODULE 1**

PIPELINE MODULE 1:
1. Load PDF
2. Extract Text
3. Break into Chunks
4. Create Embeddings
5. Store in FAISS
6. Retrieve relevant Chunks
7. Gemini Answer Questions

*FAISS (Facebook AI similarity Search) is open source library designed for similarity search and clustering of dense vector. It will act as vector database : storing embeddings and retrieving relevant onse when query is asked*

#1.  LOAD PDF

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving an image is worth 16x16.pdf to an image is worth 16x16.pdf


# 2. READ PDF FILE UPLOADED

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 10.7 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader
pdf_file = list(uploaded.keys())[0]
reader = PdfReader(pdf_file)
print("Pages:", len(reader.pages))

Pages: 22


* Imports PdfReader class from pypdf library to read PDF files.

* Gets the first uploaded file’s name from the uploaded dictionary (assuming multiple files could be uploaded).

* Creates a PdfReader object to open and read the PDF file.
* Prints the total number of pages in the PDF by checking the length of reader.pages.

# 3. EXTRACT TEXT FROM PDF make reusable function

In [ ]:
from pypdf import PdfReader
def extract_pdf_text(pdf_file):
    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()     # return text content of pdf page
    return text

* Initializes an empty string, then loops through each PDF page and appends its extracted text

In [ ]:
paper_text = extract_pdf_text(pdf_file)  #function call

In [ ]:
print(paper_text[:200])

Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16 X16 W ORDS :
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗


* Why are we extracting text from PDF why can't we use straight pdf in production?

*ans: because computers learn numbers and numerial represntation of words is embedding word on text/tokens not PDF files*


#4. TEXT CLEANING  (reusable function)
* remove multispace
* remove trailing space


In [ ]:
import re
def clean_text(text):
  text=re.sub(r'\s+',' ',text)  #remove multiple space

  text=text.strip()     #remove trail space

  return text

In [ ]:
print(paper_text[:200])

Published as a conference paper at ICLR 2021
AN IMAGE IS WORTH 16 X16 W ORDS :
TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE
Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗


In [ ]:
cleaned_text = clean_text(paper_text)

In [ ]:
print(cleaned_text[:200])

Published as a conference paper at ICLR 2021 AN IMAGE IS WORTH 16 X16 W ORDS : TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗


### Q. Why cleaning text before chunk.
*to remove extra space and formatting noise from pdf for embedding and retrieval*

## PDF TO TEXT EXTRACTION AND TEXT CLEANING IS DONE

----
----
# 5. CHUNKING (reusable function)
we do not sent entire paper to gemini we break the content into smaller pieces: chunk 1: abstract, chunk 2: intro, chunk 3: methodology, chunk4: experiments, ...

### **Why do we need Chunking ?**
*because when user asks a query we don't search entire word of paper rather than search chunks and find relevant chunk and send to gemini*

## IMPLEMENT CHUNK OVERLP
### **so that chunk contain full sentences and are not trimmed in between and remove repeated regions**

In [ ]:
def create_chunks(text, chunk_size=1000, overlap=200): # if chunk size 200-200 very small doc, 500 common, 1000 research paper and 1500+ large context model
  chunks=[]
  start=0

  while(start<len(text)):
    end=start+chunk_size
    chunk=text[start:end]
    chunks.append(chunk)

    start+= chunk_size-overlap
  return chunks

In [ ]:
chunks = create_chunks(cleaned_text)

print("Total Chunks:", len(chunks))

Total Chunks: 84


In [ ]:
print(chunks[0][-200:]) # t shows the ending portion (last 200 characters) of the first chunk,

re in place. We show that this reliance on CNNs is not necessary and a pure transformer applied directly to sequences of image patches can perform very well on image classiﬁcation tasks. When pre-trai


In [ ]:
print(chunks[1][:200])

re in place. We show that this reliance on CNNs is not necessary and a pure transformer applied directly to sequences of image patches can perform very well on image classiﬁcation tasks. When pre-trai


That means the last 200 characters of chunk 0 (800–1000) are the same as the first 200 characters of chunk 1 (800–1000). ## **SO NOT BEST APPROACH**
# problem: A sentence gets cut in the middle.

## 1. CONVERT TEXXT INTO SENTENCES



In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt_tab') #pre‑trained tokenizer model  for tokenisation

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:

from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(cleaned_text)

print("Total Sentences:", len(sentences))

Total Sentences: 600


In [ ]:
print(sentences[1])

In vision, attention is either applied in conjunction with convolutional networks, or used to replace certain components of convolutional networks while keeping their overall structure in place.


### *SOLUTION BEST CHUNKING APPROACH*
# 5 **SENTENCE BASED CHUNKING (REUSABLE FUNCTION)**

In [ ]:
def create_sentence_chunks(sentences,chunk_size=1000):
  chunks=[]
  current_chunk=""

  for sentence in sentences:
    if len(current_chunk)+len(sentence) <chunk_size:
      current_chunk+=" "+ sentence
    else:
      chunks.append(current_chunk.strip())
      current_chunk=sentence

  if current_chunk:
    chunks.append(current_chunk.strip())
  return chunks


In [ ]:
chunks=create_sentence_chunks(sentences)
print("Total Chunks:", len(chunks))

Total Chunks: 77


In [ ]:
print(chunks[0])

Published as a conference paper at ICLR 2021 AN IMAGE IS WORTH 16 X16 W ORDS : TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗, Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,† ∗equal technical contribution,†equal advising Google Research, Brain Team {adosovitskiy, neilhoulsby}@google.com ABSTRACT While the Transformer architecture has become the de-facto standard for natural language processing tasks, its applications to computer vision remain limited. In vision, attention is either applied in conjunction with convolutional networks, or used to replace certain components of convolutional networks while keeping their overall structure in place. We show that this reliance on CNNs is not necessary and a pure transformer applied directly to sequences of image patches can perform very well on image classiﬁcation tasks.


In [ ]:
print(chunks[1])

When pre-trained on large amounts of data and transferred to multiple mid-sized or small image recognition benchmarks (ImageNet, CIFAR-100, VTAB, etc. ), Vision Transformer (ViT) attains excellent results compared to state-of-the-art convolutional networks while requiring sub- stantially fewer computational resources to train.1 1 I NTRODUCTION Self-attention-based architectures, in particular Transformers (Vaswani et al., 2017), have become the model of choice in natural language processing (NLP). The dominant approach is to pre-train on a large text corpus and then ﬁne-tune on a smaller task-speciﬁc dataset (Devlin et al., 2019). Thanks to Transformers’ computational efﬁciency and scalability, it has become possible to train models of unprecedented size, with over 100B parameters (Brown et al., 2020; Lepikhin et al., 2020). With the models and datasets growing, there is still no sign of saturating performance.


Why is sentence-based chunking better than fixed-size chunking?

Answer:
It preserves semantic meaning by keeping sentences intact, improving embedding quality and retrieval accuracy.

# 6. EMBEDDING

* CONVERT TEXT INTO NUMBERS : THIS NUMERIC REPRESENTATION IS CALLED EMBEDDIN
* need why? imagine 2 sentences diff word but same meaning, a good embeddinf model places them close together in vector space.
* **Embeddings convert text into numerical vectors that capture semantic meaning, allowing similarity search and retrieval in RAG systems.**

example: sentence a: Vision Transformer uses image patches.
and sentence b: Vit process image as patch

same

**SENTENCE TRANSFORMER** They are models built on top of transformers (like BERT, RoBERTa, etc.) but fine‑tuned specifically for sentence-level embeddings.

In [ ]:
!pip install sentence-transformers

## Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer
model= SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

"sentence-transformers/all-MiniLM-L6-v2" is a lightweight, fast model that converts text chunks into meaningful vectors for semantic search.

### first embedding sample check

In [ ]:
sample_chunk=chunks[0]
embedding= model.encode(sample_chunk)

In [ ]:
print(len(embedding)) # 384 dimension vector representation of chunk 1

384


### VERIFY SIMILARITY (cosine)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
text1 = "Vision Transformer uses image patches."

text2 = "ViT processes images as patches."

emb1 = model.encode(text1)
emb2 = model.encode(text2)

In [ ]:
similarity=cosine_similarity([emb1],[emb2])
print(similarity)

[[0.56785583]]


In [ ]:
text3= "Vision market crashed today"
emb3= model.encode(text3)
similarity= cosine_similarity([emb1],[emb3])
print(similarity)

[[0.27734387]]


# 7. FAISS (Facebook AI Similarity Search)
* FAISS  is a vector database/library designed to search embeddings efficiently.

* WHEN user asks a query we create an embedding for this query and compare it against all chunk embedding.

* FAISS makes this search fast and scalable

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 104.6 MB/s eta 0:00:00


## Create embedding of all chunks

In [ ]:
chunks

['Published as a conference paper at ICLR 2021 AN IMAGE IS WORTH 16 X16 W ORDS : TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE Alexey Dosovitskiy∗,†, Lucas Beyer∗, Alexander Kolesnikov∗, Dirk Weissenborn∗, Xiaohua Zhai∗, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, Neil Houlsby∗,† ∗equal technical contribution,†equal advising Google Research, Brain Team {adosovitskiy, neilhoulsby}@google.com ABSTRACT While the Transformer architecture has become the de-facto standard for natural language processing tasks, its applications to computer vision remain limited. In vision, attention is either applied in conjunction with convolutional networks, or used to replace certain components of convolutional networks while keeping their overall structure in place. We show that this reliance on CNNs is not necessary and a pure transformer applied directly to sequences of image patches can perform very well on image classiﬁcation tasks.',
 'When pr

In [ ]:
chunk_embeddings=model.encode(chunks)
print(chunk_embeddings.shape)

(77, 384)


77 document chunks, 384 dimensional embedding vector for each chunk

## CREATE FAISS INDEX

In [ ]:
import faiss
import numpy as np

In [ ]:
embeddings= np.array(chunk_embeddings,dtype=np.float32)

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

In [ ]:
print(dimension)

384


In [ ]:
index.add(embeddings)

In [ ]:
print(index.ntotal, "vectors stored successfully")

77 vectors stored successfully


FAISS stores numerical vector embeddings, not raw text.

# **8. RETRIEVE RELEVANT CHUNKS**
## Semantic Search reusable function

In [ ]:
def search(query,model,index,chunks,k=3):
  query_embedding=model.encode([query])
  distances,indices=index.search(np.array(query_embedding,dtype=np.float32),k)
  result=[]
  for idx in indices[0]:
    result.append(chunks[idx])
  return result

* takes query, model, FAISS index, chunks, and k results

* Encode query → query_embedding (vector representation)
* Run FAISS index.search(...) → get top‑k distances + indices.
* Loop through each neighbor index

In [ ]:
query = "What dataset was used?"
results = search(query,model,index,chunks,k=3)

In [ ]:
for i, chunk in enumerate(results):

    print(f"\n----- Result {i+1} -----\n")

    print(chunk[:500])


----- Result 1 -----

To explore model scalability, we use the ILSVRC-2012 ImageNet dataset with 1k classes and 1.3M images (we refer to it as ImageNet in what follows), its superset ImageNet-21k with 21k classes and 14M images (Deng et al., 2009), and JFT (Sun et al., 2017) with 18k classes and 303M high-resolution images. We de-duplicate the pre-training datasets w.r.t. the test sets of the downstream tasks following Kolesnikov et al. (2020). We transfer the models trained on these dataset to several benchmark task

----- Result 2 -----

These values correspond to Figure 3 in the main text. Models are ﬁne-tuned at 384 resolution. Note that the ImageNet results are computed without additional techniques (Polyak averaging and 512 resolution images) used to achieve results in Table 2.

----- Result 3 -----

The larger model, ViT-H/14, further improves the performance, especially on the more challenging datasets – ImageNet, CIFAR-100, and the VTAB suite. Interestingly, this 5Published a

k in your function is simply the number of nearest neighbors

Default k=3 → the function will fetch the top 3 most similar chunks to the query.



**What is retrieved from FAISS?**

Answer:

FAISS returns the indices of the most similar embedding vectors. Those indices are used to retrieve the corresponding text chunks.

# 7. Add Gemini

This is the essence of RAG (Retrieval-Augmented Generation).

1. INSTALL GEMINI SDK

In [ ]:
!pip install -q google-generativeai

2. CONFIGURE xxxxxxxxxxxxxxxxxxxxxxxxx  API KEY

In [ ]:
import google.generativeai as genai

genai.configure(api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


3. LOAD GEMINI

In [ ]:
model_gemini = genai.GenerativeModel(
    "gemini-2.5-flash"
)

4. CONTEXT FROM RETRIEVED CHUNKS
*Context Contains the most relevant chunks from the paper*

In [ ]:
results = search(query,model,index,chunks,k=3)

In [ ]:
context = "\n\n".join(results)

5. BUILD THE PROMPT:
* *prompt engineering*

In [ ]:
prompt = f"""
You are a research paper assistant.

Answer ONLY using the provided context.

If the answer is not present in the context,
say:

'I could not find that information in the paper.'

Context:
{context}

Question:
{query}

Answer:
"""

*I used this prompt because without instructions gemini may use outside knowledge, with prompt retrieved context only* **THIS MITIGATES HALLUCINATIONS**

In [ ]:
response = model_gemini.generate_content(prompt)
print(response.text)

The datasets used are:
*   ILSVRC-2012 ImageNet (referred to as ImageNet)
*   ImageNet-21k
*   JFT
*   ImageNet ReaL
*   CIFAR-10/100
*   Oxford-IIIT Pets
*   Oxford Flowers-102
*   VTAB (19 tasks)


In [ ]:
query = "What is the main contribution of the paper?"
response = model_gemini.generate_content(prompt)
print(response.text)

The datasets used are:
*   ILSVRC-2012 ImageNet
*   ImageNet-21k
*   JFT
*   ImageNet (on original validation labels and ReaL labels)
*   CIFAR-10/100
*   Oxford-IIIT Pets
*   Oxford Flowers-102
*   VTAB suite


#### GEMINI IS WORKING BUT RETURNED WRONG CHUNKS

because the issue is retrieval quality as **The LLM can only answer from the retrieved chunks.
If retrieval is poor, the final answer will also be poor.**
Garbage In → Garbage Out

### will debug it later
# create reusable fucntions

* generate_summary()
* extract_methodology()
* extract_datasets()
* extract_results()
* extract_limitations()
* extract_future_work()

## 1. generate summary function

In [ ]:
def generate_summary(text):

    prompt = f"""
    You are a research paper assistant.

    Summarize the following research paper.

    Focus on:
    - Problem Statement
    - Proposed Approach
    - Key Contributions
    - Results

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)

    return response.text

In [ ]:
summary = generate_summary(cleaned_text)
print(summary)

## Research Paper Summary: An Image Is Worth 16x16 Words: Transformers For Image Recognition At Scale

This paper introduces the Vision Transformer (ViT), a novel approach that applies a standard Transformer architecture directly to images for classification tasks, eschewing the widespread reliance on convolutional neural networks (CNNs).

### Problem Statement

While Transformer architectures have become the de-facto standard in Natural Language Processing (NLP) due to their scalability and efficiency, their application to computer vision (CV) has been limited. Existing approaches in CV either integrate attention mechanisms with CNNs or replace specific CNN components while retaining the overall convolutional structure. These specialized attention patterns often lack the computational efficiency to scale effectively on modern hardware, leaving classic ResNet-like architectures dominant in large-scale image recognition. The core problem ViT addresses is whether a "pure" Transformer, wi

# 2. METHODOLOGY EXTRACTOR

In [ ]:
def extract_methodology(text):

    prompt = f"""
    Analyze the research paper.

    Extract only the methodology.

    Include:
    - Model Architecture
    - Training Procedure
    - Workflow
    - Important Techniques

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)
    return response.text

In [ ]:
methodology = extract_methodology(cleaned_text)
print(methodology)

The methodology of the Vision Transformer (ViT) paper is detailed below, covering its architecture, training, workflow, and key techniques.

---

### Methodology

The Vision Transformer (ViT) adapts the standard Transformer architecture from NLP directly to image classification tasks, with minimal modifications.

#### Model Architecture

The core idea is to treat image patches as tokens in a sequence, similar to words in NLP.

1.  **Image to Patches:**
    *   An input image `x ∈ R^(H×W×C)` (Height, Width, Channels) is reshaped into a sequence of flattened 2D patches `xp ∈ R^(N×(P²·C))`.
    *   `(P, P)` is the resolution of each square image patch.
    *   `N = HW/P²` is the resulting number of patches, which serves as the effective input sequence length for the Transformer.

2.  **Patch Embedding:**
    *   Each flattened patch `xp_i` is linearly projected to a constant latent vector size `D` using a trainable linear projection matrix `E ∈ R^((P²·C)×D)`. This generates the patch embe

# 3. Dataset Extractor function

In [ ]:
def extract_datasets(text):

    prompt = f"""
    Identify all datasets used in this research paper.

    For each dataset provide:
    - Dataset Name
    - Purpose

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)
    return response.text

In [ ]:
extractor=extract_datasets(cleaned_text)
print(extractor)

Here are the datasets used in the research paper, along with their purposes:

1.  **Dataset Name:** ILSVRC-2012 ImageNet (referred to as ImageNet)
    *   **Purpose:**
        *   Mid-sized dataset for pre-training.
        *   Benchmark for fine-tuning and evaluation, particularly to show ViT's performance when trained on "insufﬁcient amounts of data" compared to ResNets.
        *   Used for few-shot linear accuracy evaluation.
        *   Comprises 1k classes and 1.3M images.

2.  **Dataset Name:** ImageNet-21k
    *   **Purpose:**
        *   Large public dataset for pre-training ViT models.
        *   Used to demonstrate that "large scale training trumps inductive bias" for Vision Transformers.
        *   Serves as a stepping stone between ImageNet and JFT-300M in terms of dataset size for pre-training experiments.
        *   Comprises 21k classes and 14M images.

3.  **Dataset Name:** JFT-300M
    *   **Purpose:**
        *   Very large, in-house dataset for pre-training Visio

# 4. Results Extractor Function

In [ ]:
def extract_results(text):

    prompt = f"""
    Extract the experimental results from this paper.

    Include:
    - Metrics
    - Accuracy
    - Comparisons
    - Key Findings

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)
    return response.text

In [ ]:
results = extract_results(cleaned_text)
print(results)

The experimental results from the paper "AN IMAGE IS WORTH 16X16 WORDS: TRANSFORMERS FOR IMAGE RECOGNITION AT SCALE" are summarized below:

---

### Experimental Results

**Metrics:**
The primary metrics reported are **fine-tuning accuracy** (Top-1 accuracy) on various downstream image classification benchmarks and **linear few-shot accuracy** for specific data requirement studies. Computational cost is measured in **TPUv3-core-days** for pre-training efficiency comparisons.

**Evaluated Benchmarks:**
*   **ImageNet** (1k classes, 1.3M images)
*   **ImageNet-ReaL** (cleaned-up ImageNet labels)
*   **CIFAR-10**
*   **CIFAR-100**
*   **Oxford-IIIT Pets**
*   **Oxford Flowers-102**
*   **VTAB** (a suite of 19 diverse classification tasks, divided into Natural, Specialized, and Structured groups)

**Pre-training Datasets:**
*   **ImageNet**
*   **ImageNet-21k** (21k classes, 14M images)
*   **JFT-300M** (18k classes, 303M high-resolution images)

**Accuracy (ViT-H/14 pre-trained on JFT-300

# 5. Limitations Extractor

In [ ]:
def extract_limitations(text):

    prompt = f"""
    Identify limitations mentioned in the paper.

    If limitations are not explicitly stated,
    infer possible limitations based on the paper.

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)
    return response.text

In [ ]:
limitations=extract_limitations(cleaned_text)
print(limitations)

Based on the paper, here are the limitations mentioned or inferable for the Vision Transformer (ViT):

**Explicitly Stated Limitations:**

1.  **Requirement for Large-Scale Pre-training Data:** ViT models "do not generalize well when trained on insufficient amounts of data" and "yield modest accuracies of a few percentage points below ResNets of comparable size" when trained on mid-sized datasets (like ImageNet) without strong regularization. They only attain excellent results "When pre-trained on large amounts of data."
2.  **Lack of Inductive Biases:** Transformers "lack some of the inductive biases inherent to CNNs, such as translation equivariance and locality," which contributes to their poor generalization on smaller datasets. The 2D structure of images is used "very sparingly" (only during patching and position embedding adjustment for different resolutions).
3.  **Overfitting on Smaller Datasets:** "Vision Transformers overﬁt more than ResNets with comparable computational cost

# 6. FUTURE WORK EXTRACTOR

In [ ]:
def extract_future_work(text):

    prompt = f"""
    Extract future work and research directions
    mentioned in the paper.

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)
    return response.text

In [ ]:
future_work=extract_future_work(cleaned_text)
print(future_work)

The paper mentions the following future work and research directions:

1.  **Self-supervised pre-training for Vision Transformers:** The authors state that "self-supervised ViT holds promise for the future" and dedicate a section (4.6) to it, indicating this as an area for further investigation.
2.  **Further analysis of few-shot properties of ViT:** The paper explicitly states that "Further analysis of few-shot properties of ViT is an exciting direction of future work."
3.  **Continued scaling efforts for Vision Transformers:** The authors note that "Vision Transformers appear not to saturate within the range tried, motivating future scaling efforts." This suggests exploring even larger models or datasets.


# 7. interview questions GENERATOR

In [ ]:
def generate_interview_questions(text):

    prompt = f"""
    Generate 10 technical interview questions
    and answers from this research paper.

    Include:
    - Conceptual Questions
    - Architecture Questions
    - Methodology Questions
    - Practical Questions

    Paper:
    {text[:30000]}
    """

    response = model_gemini.generate_content(prompt)
    return response.text

In [ ]:
questions=generate_interview_questions(cleaned_text)
print(questions)

Here are 10 technical interview questions and answers based on the provided research paper, categorized as requested:

---

### Conceptual Questions

**1. Question:** What is the core innovation of the Vision Transformer (ViT) proposed in this paper, and how does it fundamentally differ from prior approaches of applying attention to computer vision?

**Answer:** The core innovation of ViT is demonstrating that a *pure, standard Transformer* architecture, applied directly to sequences of image patches, can perform very well on image classification tasks, especially when pre-trained on large amounts of data. This fundamentally differs from prior approaches, which typically applied attention *in conjunction with* convolutional networks or replaced specific CNN components, thereby retaining their overall convolutional structure and inductive biases. ViT explicitly shows that reliance on CNNs for vision tasks is not necessary.

**2. Question:** Explain the concept of "inductive bias" in the

1. User Uploads PDF
        
2. PDF Extraction
        
3. Text Cleaning
        
4. Chunking
        
5. Sentence Transformer Embeddings
        
6. FAISS Vector Store
        
7. Semantic Retrieval
        
8. Gemini
        
9. Generated Outputs

---
---


# **ACTUAL RAG QA FUNCTION**

1. User Question
2. Embedding
3. FAISS search
4. Retrieve Top Chunks
5. GEMINi answer



In [ ]:
def ask_question(query, model, index, chunks, model_gemini, k=5):

    # Convert question to embedding
    query_embedding = model.encode([query])

    # Search FAISS
    distances, indices = index.search(
        np.array(query_embedding, dtype=np.float32),
        k
    )

    # Retrieve chunks
    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    # Build context
    context = "\n\n".join(retrieved_chunks)

    # Prompt for mitigating hallucination
    prompt = f"""
    You are a research paper assistant.
    Answer ONLY using the provided context.
    If the answer is not present in the context,
    reply:
    "I could not find that information in the paper."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    response = model_gemini.generate_content(prompt)
    return response.text

##### PROVED THAT MODEL IS NOT HALLUCINATING because this function converted question to embed and searcched FAISS and got top 3 chunks and sent to gemini : **GEMINI ONLY ANSWERS FROM TOP 3 CHUNKS** if not states I could not find that information in the paper.

## DEBUG

In [ ]:
query = "What methodology is proposed?"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding, dtype=np.float32),
    5
)

for i, idx in enumerate(indices[0]):
    print(f"\n===== Retrieved Chunk {i+1} =====\n")
    print(chunks[idx][:1000])


===== Retrieved Chunk 1 =====

Our initial experiments show improvement from self-supervised pre-training, but there is still large gap between self-supervised and large-scale supervised pre- training. Finally, further scaling of ViT would likely lead to improved performance. ACKNOWLEDGEMENTS The work was performed in Berlin, Z ¨urich, and Amsterdam. We thank many colleagues at Google for their help, in particular Andreas Steiner for crucial help with the infrastructure and the open- source release of the code; Joan Puigcerver and Maxim Neumann for help with the large-scale training infrastructure; Dmitry Lepikhin, Aravindh Mahendran, Daniel Keysers, Mario Luˇci´c, Noam Shazeer, Ashish Vaswani, and Colin Raffel for useful discussions. REFERENCES Samira Abnar and Willem Zuidema. Quantifying attention ﬂow in transformers. In ACL, 2020. Philip Bachman, R Devon Hjelm, and William Buchwalter. Learning representations by maximizing mutual information across views. In NeurIPS, 2019.

===== R

NONE IS RELEVANT

In [ ]:
print(
    ask_question(
        "What is Vision Transformer?",
        model,
        index,
        chunks,
        model_gemini,
        k=5
    )
)

A Vision Transformer (ViT) is a pure transformer applied directly to sequences of image patches for image classification tasks. It works by splitting an image into fixed-size patches, linearly embedding each of them, adding position embeddings, and feeding the resulting sequence of vectors to a standard Transformer encoder. For classification, an extra learnable "classification token" is added to the sequence.


# SAVING VECTOR DATABASE

In [ ]:
faiss.write_index(
    index,
    "paper_index.faiss"
)

# SAVING CHUNKS

In [ ]:
import pickle

with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

# SAVE EMBEDDINGS

In [ ]:
import numpy as np

np.save(
    "embeddings.npy",
    chunk_embeddings
)

# ENGINEERING DONE

-------
-----

1. User Uploads PDF
2. Process PDF     
3. Create Chunks
4. Create Embeddings
5. Build FAISS
6. User Clicks Buttons
7. Your Existing Functions Run
8. Results Displayed

In [ ]:
!pip install -q gradio pypdf nltk sentence-transformers faiss-cpu google-generativeai

In [ ]:
!python app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3bca142526319a6387.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Loading weights: 100% 103/103 [00:00<00:00, 1449.68it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/app.py:45: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com